# Exercise solutions — Semantic segmentation — U-Net

**DeepVision Masterclass · Solutions companion**

These are worked reference solutions for the exercises in the matching numbered lesson. Run the parent lesson first when variables or datasets are reused. Dataset-specific experiments are left as clearly marked tasks; no performance scores are fabricated.

## Exercise 1 — Complete paired image/mask loader
This generic loader assumes masks are single-channel integer label images. Adapt palette decoding for indexed-color PNG masks.

In [ ]:
from PIL import Image
import numpy as np, torch
from torch.utils.data import Dataset
class PairedSegmentationDataset(Dataset):
    def __init__(self,image_paths,mask_paths,size=(256,256)):
        assert len(image_paths)==len(mask_paths)
        self.image_paths,self.mask_paths,self.size=image_paths,mask_paths,size
    def __len__(self): return len(self.image_paths)
    def __getitem__(self,i):
        image=Image.open(self.image_paths[i]).convert('RGB').resize(self.size,Image.Resampling.BILINEAR)
        mask=Image.open(self.mask_paths[i]).convert('L').resize(self.size,Image.Resampling.NEAREST)
        x=torch.from_numpy(np.asarray(image).copy()).permute(2,0,1).float()/255.
        y=torch.from_numpy(np.asarray(mask).copy()).long()
        return x,y
# Ensure mask pixel values are contiguous class IDs (0..C-1); remap source labels explicitly if needed.

## Exercise 2 — Class-weighted cross-entropy and Dice loss

In [ ]:
import torch.nn.functional as F
def multiclass_dice_loss(logits,target,n_classes,eps=1e-6):
    probs=logits.softmax(1); onehot=F.one_hot(target,n_classes).permute(0,3,1,2).float()
    dims=(0,2,3); intersection=(probs*onehot).sum(dims); denom=probs.sum(dims)+onehot.sum(dims)
    dice=(2*intersection+eps)/(denom+eps)
    return 1-dice.mean()
class_weights=torch.tensor([.3,1.0],device=device) # derive from training-set pixel frequencies, not test masks
ce=nn.CrossEntropyLoss(weight=class_weights)
# loss=ce(logits,masks)+multiclass_dice_loss(logits,masks,n_classes=2)

## Exercise 3 — Mean IoU

In [ ]:
def mean_iou(logits,target,n_classes,include_background=False):
    pred=logits.argmax(1); scores=[]
    for c in range(n_classes):
        p=(pred==c); t=(target==c); union=(p|t).sum()
        if union>0: scores.append((p&t).sum().float()/union)
    if not include_background: scores=scores[1:]
    return torch.stack(scores).mean().item() if scores else float('nan')

## Exercise 4 — Oxford-IIIT Pet
The dataset provides trimap segmentation labels. Convert trimap values to class IDs consistently, and decide whether boundaries are a separate class. Do not confuse the image classification label with the pixel mask.

In [ ]:
import tensorflow_datasets as tfds
# Optional alternative dataset access:
# ds, info = tfds.load('oxford_iiit_pet:3.*.*', with_info=True, as_supervised=False)
# Inspect `segmentation_mask` values and documentation before remapping labels.

## Exercise 5 — DeepLabV3 comparison
Use torchvision DeepLabV3 with a pretrained backbone, replace the final classifier for your class count, and compare mean IoU/Dice on the exact same split. For large images, tiled inference must include overlap and a blending strategy.

## Reporting template
Record the dataset/source and license, split strategy, seed(s), preprocessing, model configuration, optimizer, epochs, best validation checkpoint, final test metrics, hardware and runtime. Replace illustrative code comments with measured results from your own run.